# 02 · The Right Model for the Job

**Module 09 · Cost Optimization**

You would not send a senior engineer to answer "what are your opening hours?". A **router** looks at each email first and picks the model: the cheap one for easy emails, the strong one for hard ones.

Two routers, so we can compare them:

| Router | How it decides | Cost | Understands the email? |
|---|---|---|---|
| **Rules** | word count, question marks, a few keywords | free, instant | no, only the surface |
| **LLM** | the cheap model reads it and answers "easy" or "hard" | a tiny call | yes |

In [1]:
import inbox
import llm
import router
import strategies

print("Ready")

Ready


*From `router.py`:*

```python
HARD_WORDS = ["lawyer", "exception", "frustrated", "angry", "card number", "how much more", "in total", "after the"]


def route_by_rules(email):
    """Plain rules. Returns 'easy' or 'hard'. Free, and it never makes a model call."""
    text = email.lower()

    if len(text.split()) > 20:          # a long email usually asks for several things
        return "hard"
    if text.count("?") > 1:             # several questions
        return "hard"
    for word in HARD_WORDS:
        if word in text:
            return "hard"
    return "easy"
```

*From `router.py`:*

```python
class Difficulty(BaseModel):
    difficulty: Literal["easy", "hard"]


ROUTER_INSTRUCTIONS = (
    "You sort customer emails for an online shop. "
    "Answer 'easy' if the email asks ONE simple question about shipping, payment, returns or warranty "
    "that a single line of a policy answers. "
    "Answer 'hard' if it asks several things at once, needs a calculation, asks for an exception, "
    "mentions a lawyer, a card number or an angry complaint, or needs two rules to be combined."
)


def route_by_llm(email):
    """The cheap model decides. Returns (difficulty, cost in dollars, seconds)."""
    started = time.time()
    response = llm.client.chat.completions.parse(
        model=llm.SMALL,
        messages=[
            {"role": "system", "content": ROUTER_INSTRUCTIONS},
            {"role": "user", "content": email},
        ],
        response_format=Difficulty,
        max_tokens=20,
        temperature=0,
    )
    seconds = time.time() - started

    usage = response.usage
    cost = llm.cost_of(llm.SMALL, usage.prompt_tokens, 0, usage.completion_tokens)
    return response.choices[0].message.parsed.difficulty, cost, seconds
```

## 1. How good is each router?
We know the true difficulty of every email in the test set (the `kind` label), so we can **measure the router** like anything else:

In [2]:
def router_report(name, decide):
    """Count how often a router agrees with our label, and where it is wrong."""
    right = 0
    too_cheap = []      # a hard email sent to the small model: a risk of a cheap, WRONG answer
    too_costly = []     # an easy email sent to the big model: money wasted
    total_cost = 0.0

    for item in inbox.EMAILS:
        label, cost = decide(item["email"])
        total_cost = total_cost + cost
        if label == item["kind"]:
            right = right + 1
        elif label == "easy":
            too_cheap.append(item["email"])
        else:
            too_costly.append(item["email"])

    print(name, ":", right, "of", len(inbox.EMAILS), "correct.  Router cost: $" + format(total_cost, ".5f"))
    print("   hard emails sent to the cheap model:", len(too_cheap))
    for email in too_cheap:
        print("      -", email[:80])
    print("   easy emails sent to the strong model:", len(too_costly))
    for email in too_costly:
        print("      -", email[:80])


def rules_decide(email):
    return router.route_by_rules(email), 0.0


def llm_decide(email):
    label, cost, seconds = router.route_by_llm(email)
    return label, cost


router_report("Rules router", rules_decide)
print()
router_report("LLM router  ", llm_decide)

Rules router : 25 of 28 correct.  Router cost: $0.00000
   hard emails sent to the cheap model: 3
      - I ordered a $40 item with express shipping. What will the shipping cost me, and 
      - My phone stopped working after 14 months. Is it still under warranty, and can I 
      - My $300 television arrived 10 days ago and I changed my mind. Please refund the 
   easy emails sent to the strong model: 0



LLM router   : 27 of 28 correct.  Router cost: $0.00071
   hard emails sent to the cheap model: 0
   easy emails sent to the strong model: 1
      - Does the warranty cover a phone that I dropped?


Read the two kinds of mistake, because they cost different things:

- A hard email sent to the **cheap** model risks a **wrong answer**: the customer pays for our saving.
- An easy email sent to the **strong** model only **wastes money**.

So when in doubt, a router should lean towards the strong model. Rules are free but blind to meaning. The LLM router costs a fraction of a cent per email, and it is usually the better one. We use it from here on.

## 2. Three ways to run the desk, on the same 28 emails
- **always strong**: `gpt-4o` for everything
- **always cheap**: `gpt-4o-mini` for everything
- **routed**: the LLM router decides, then the chosen model answers (the router's call is **included** in the cost)

(This runs 84 calls and takes about a minute and a half.)

*From `strategies.py`:*

```python
def evaluate(strategy, items):
    """Run a strategy on every item and measure cost, quality and speed.
    Quality = how many replies contained EVERY required fact."""
    total_cost = 0.0
    total_seconds = 0.0
    perfect = 0
    sources = {}
    misses = []

    for item in items:
        result = strategy(item["email"])
        score = inbox.score_reply(result["reply"], item["facts"])

        total_cost = total_cost + result["cost"]
        total_seconds = total_seconds + result["seconds"]
        if score == 1.0:
            perfect = perfect + 1
        else:
            misses.append({"email": item["email"], "reply": result["reply"], "source": result["source"]})

        sources[result["source"]] = sources.get(result["source"], 0) + 1

    count = len(items)
    return {
        "emails": count,
        "cost": total_cost,
        "cost_per_1000": total_cost / count * 1000,
        "perfect": perfect,
        "avg_seconds": total_seconds / count,
        "sources": sources,
        "misses": misses,
    }
```

In [3]:
results = {}
results["always strong (gpt-4o)"] = strategies.evaluate(strategies.always_big, inbox.EMAILS)
results["always cheap (gpt-4o-mini)"] = strategies.evaluate(strategies.always_small, inbox.EMAILS)
results["routed"] = strategies.evaluate(strategies.routed, inbox.EMAILS)

baseline = results["always strong (gpt-4o)"]["cost_per_1000"]
print("Setup".ljust(28) + "$ per 1,000".rjust(12) + "All facts".rjust(11) + "Avg seconds".rjust(13) + "Saved".rjust(8))
print("-" * 72)
for name in results:
    row = results[name]
    saved = (1 - row["cost_per_1000"] / baseline) * 100
    print(name.ljust(28) + format(row["cost_per_1000"], ".2f").rjust(12)
          + (str(row["perfect"]) + "/" + str(row["emails"])).rjust(11)
          + format(row["avg_seconds"], ".1f").rjust(13) + (format(saved, ".0f") + "%").rjust(8))

Setup                        $ per 1,000  All facts  Avg seconds   Saved
------------------------------------------------------------------------
always strong (gpt-4o)              1.91      28/28          0.8      0%
always cheap (gpt-4o-mini)          0.12      26/28          0.8     94%
routed                              1.34      28/28          1.4     30%


In [4]:
for name in results:
    print("====", name, " where the replies came from:", results[name]["sources"])
    for miss in results[name]["misses"]:
        print("   MISSED A FACT:", miss["email"][:70])
        print("                  ", miss["reply"][:160].replace("\n", " "))

==== always strong (gpt-4o)  where the replies came from: {'gpt-4o': 28}
==== always cheap (gpt-4o-mini)  where the replies came from: {'gpt-4o-mini': 28}
   MISSED A FACT: I ordered three items: $19.99, $14.50 and $9.00. Will I pay for standa
                   Yes, you will get free standard shipping since your order total is over $50. For the opened, not faulty $14.50 item, a 10% restocking fee applies, which is $1.4
   MISSED A FACT: My order is $49.50. How much more would express shipping cost me compa
                   Express shipping costs $12, while standard shipping is free for orders over $50. You would need to spend an additional $0.50 to qualify for free standard shippi
==== routed  where the replies came from: {'gpt-4o-mini': 11, 'gpt-4o': 17}


## 3. Reading the table honestly
Numbers vary a little from run to run (models are not fully deterministic), but the **shape** is stable:

1. **Always strong** is the quality ceiling, and the most expensive.
2. **Always cheap** saves about 94%, and may miss an arithmetic fact. Whether that is acceptable is a **business decision** (how bad is a wrong shipping price?), not a technical one.
3. **Routed** keeps the quality of the strong model for a lot less. It does **not** reach the cheap model's price, for two reasons you can see in the table: a large share of our test emails (16 of 28) are "hard", so most of them go to the strong model, and the router's own call adds a little cost and about half a second.

> **An honest limit.** Our router is cautious: it sends many emails to the strong model that the cheap one could have answered. A smarter design is a **cascade**: try the cheap model first, check the answer (with a rule or a second cheap check), and only **escalate** if it fails. It saves more, and it is harder to build well. Know the name; build it when your measurements say it pays.

## What you learned
- A **router** picks a model per request. Measure it against emails with a known difficulty.
- The two router mistakes cost different things: wrong answers versus wasted money. Lean towards the strong model.
- The router is a model call too, so **its cost counts**.
- The cheapest model that passes **your** tests is the right model. Test, do not guess.

**Next:** stop paying for answers we already have.